In [55]:
# SEE https://irsa.ipac.caltech.edu/docs/program_interface/sia_allwise_atlas.html

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.utils.data import download_file
from astropy.io import fits

import pyvo

In [56]:
meta = pd.read_csv('ecle-metadata.csv')

meta

,name,ra,ra_unit,dec,dec_unit,redshift,coord_and_redshift_refs,radio_data,radio_data_comment
0,2019qiz,04:46:38.880,hour,-10:13:35.90,degree,0.0151,TNS,True,"Kate (very good radio dataset, still need to a..."
1,SDSS_J0748,07:48:20.6668,hour,+47:12:14.2648,degree,0.0616,Callow+2024,True,"Kate, VLASS"
2,SDSS_J0807,08:07:27.3157,hour,+14:05:37.0892,degree,0.0738,Callow+2024,True,VLASS
3,SDSS_J0938,09:38:01.6376,hour,+13:53:17.0423,degree,0.1010,Callow+2024,True,"Kate, VLASS"
4,SDSS_J0952,09:52:09.5629,hour,+21:43:13.2979,degree,0.0795,Callow+2024,True,"Kate, VLASS"
5,SDSS_J1055,10:55:26.4177,hour,+56:37:13.1010,degree,0.0740,Callow+2024,True,"Kate, VLASS"
6,SDSS_J1207,12:07:19.8102,hour,+24:11:55.8789,degree,0.0503,Callow+2024,True,VLASS
7,SDSS_J1241,12:41:34.2561,hour,+44:26:39.2636,degree,0.0419,Callow+2024,True,"Kate, VLASS"
8,SDSS_J1247,12:47:26.3719,hour,+07:05:25.0809,degree,0.1040,Callow+2024,True,VLASS
9,SDSS_J1342,13:42:44.4150,hour,+05:30:56.1451,degree,0.0365,Callow+2024,True,"Kate, VLASS"


In [24]:
service = pyvo.dal.TAPService('https://irsa.ipac.caltech.edu/TAP')

db_table = 'allwise_p3as_psd'
search_radius_arcseconds = 5 # in arcseconds
search_radius_degree = search_radius_arcseconds / 3600

for ii, row in meta.iterrows():
    print()
    print('################################################')
    print()
    print(row['name'])
    
    coord = SkyCoord(row.ra, row.dec, unit=('hour', 'deg'))
    ra, dec = coord.ra.value, coord.dec.value
    
    query = f'''
    SELECT *
    FROM {db_table}
    WHERE CONTAINS(POINT('ICRS',ra, dec), CIRCLE('ICRS',{ra},{dec},{search_radius_degree}))=1
    '''
    
    res = service.run_async(query)
    tab = res.to_table()
    print(tab)


################################################

2019qiz
designation  ra dec sigra  sigdec sigradec ...  x   y   z  spt_ind htm20
            deg deg arcsec arcsec  arcsec  ...                          
----------- --- --- ------ ------ -------- ... --- --- --- ------- -----

################################################

SDSS_J0748
    designation          ra         dec     ...  spt_ind      htm20     
                        deg         deg     ...                         
------------------- ----------- ----------- ... --------- --------------
J074820.66+471214.2 117.0860950  47.2039675 ... 223021103 16257754802580

################################################

SDSS_J0807
    designation          ra         dec     ...  spt_ind      htm20     
                        deg         deg     ...                         
------------------- ----------- ----------- ... --------- --------------
J080727.31+140537.1 121.8638140  14.0936458 ... 222112301 16040751174838

#############

# Now we try to pull the images from unWISE

This url is the SIA2 url:  https://irsa.ipac.caltech.edu/SIA?COLLECTION=wise_unwise&



In [67]:
sia2_url = 'https://irsa.ipac.caltech.edu/ibe/sia/wise/allwise/p3am_cdd?'#'https://irsa.ipac.caltech.edu/SIA?COLLECTION=wise_unwise&'

service = pyvo.dal.SIAService(sia2_url)

for ii, row in meta.iterrows():
    print()
    print('################################################')
    print()
    print(row['name'])
    
    coord = SkyCoord(row.ra, row.dec, unit=('hour', 'deg'))
    
    img_meta_table = service.search(pos=coord, size=5.0*u.arcsec)
    
    print(img_meta_table)
    
    outdir = os.path.join(os.getcwd(), 'wise-fits-data', row['name'])
    for row in img_meta_table:
        fname = download_file(row.getdataurl(), cache='update', show_progress=True)
        fitsfile = fits.open(fname)
        
        filename = f"{row.title.replace(' ','_')}.fits"
        outname = os.path.join(outdir,filename)
        fitsfile.writeto(outname, overwrite=True)
        
        print(f'Saved to {outname}')


################################################

2019qiz
<Table length=4>
      sia_title        ...    coadd_id  
                       ...              
        object         ...     object   
---------------------- ... -------------
W1 Coadd 0723m107_ac51 ... 0723m107_ac51
W2 Coadd 0723m107_ac51 ... 0723m107_ac51
W3 Coadd 0723m107_ac51 ... 0723m107_ac51
W4 Coadd 0723m107_ac51 ... 0723m107_ac51
Saved to /home/nfranz/radio-ecle/wise-fits-data/2019qiz/W1_Coadd_0723m107_ac51.fits


URLError: <urlopen error timed out>